# Dengo interactive explorer

Set initial conditions with the sliders below and see the primordial H/He/H2 chemistry solve update immediately. Two modes:

- **Cool at constant density** -- a fixed-density box radiating away its initial thermal energy (like the start of `examples/primordial_network.py`).
- **Free-fall collapse** -- gas in gravitational free-fall from the chosen starting density up to a target density (like `examples/free_fall_collapse.py`), the regime this project targets: H2 formation heating (4.48 eV/molecule) around 1500-2500 K near $n \sim 10^{15}\,\mathrm{cm^{-3}}$.

This uses one persistent `Solver` handle and `step_inplace()`/`solver.state` (zero Python marshaling per call -- see NOTES.md) the whole notebook runs, so even a few-thousand-step trajectory redraws in well under a second on each slider release.

Run with (from the repo root): `uv run --group notebook jupyter lab examples/interactive_explorer.ipynb`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from dengo.primordial_network import build_network, build_solver

KB = 1.3806504e-16      # erg/K
MH = 1.67e-24           # g
G_GRAV = 6.674e-8       # cm^3 g^-1 s^-2

OUTPUT_DIR = "_interactive_explorer_build"

network = build_network()
species_names = sorted(s.name for s in network.required_species)
mod = build_solver(network, OUTPUT_DIR)
solver = mod.Solver(1)  # kept open for the notebook's whole lifetime
print("Solver ready:", mod.SPECIES_NAMES)

In [ ]:
def make_ics(nH, T, x_ion, x_h2):
    """A mostly-neutral (or partly-ionized, per x_ion) primordial gas at
    number density nH (cm^-3), with an initial gas energy corresponding
    to (roughly) temperature T -- the solver's own Newton iteration
    refines T self-consistently from here."""
    return {
        "H_1": nH * 0.76 * (1.0 - x_ion),
        "H_2": nH * 0.76 * x_ion,
        "He_1": nH * 0.24 / 4.0 * (1.0 - x_ion),
        "He_2": 0.0,
        "He_3": nH * 0.24 / 4.0 * x_ion,
        "H_m0": nH * 1e-12,
        "H2_1": nH * x_h2,
        "H2_2": nH * 1e-12,
        "de": nH * 0.76 * x_ion + 2.0 * nH * 0.24 / 4.0 * x_ion,
        "ge": 1.5 * KB * T / MH,
    }


def set_state(ics):
    for name in species_names:
        solver.state[:, mod.SPECIES_INDEX[name]] = ics[name]


def state_scalar():
    return {name: solver.state[0, mod.SPECIES_INDEX[name]] for name in species_names}


def h2_fraction(s):
    total_H = s["H_1"] + s["H_2"] + s["H_m0"] + 2.0 * (s["H2_1"] + s["H2_2"])
    return 2.0 * s["H2_1"] / total_H


def thermodynamic_gamma(s):
    """Composition-weighted adiabatic index for the compressional-
    heating step in free-fall mode -- same simplification (H2 at a fixed
    gamma=7/5) as examples/free_fall_collapse.py."""
    n_total = sum(s[n] for n in species_names if n != "ge")
    inv_gm1_sum = 0.0
    for n in species_names:
        if n == "ge":
            continue
        gamma_i = 7.0 / 5.0 if n in ("H2_1", "H2_2") else 5.0 / 3.0
        inv_gm1_sum += s[n] / (gamma_i - 1.0)
    return n_total / inv_gm1_sum + 1.0


def run_constant_density(nH, T, x_ion, x_h2, log_dtf, safety_factor=0.1, max_steps=2000):
    """Cool at fixed density from the given ICs for up to 10**log_dtf
    seconds, with dt set each step from the current cooling time (same
    recipe as run_dengo.py's cooldown phase)."""
    set_state(make_ics(nH, T, x_ion, x_h2))
    dtf_total = 10.0 ** log_dtf
    t = 0.0
    t_hist, T_hist, h2_hist = [], [], []
    for _ in range(max_steps):
        s = state_scalar()
        rhs = solver.evaluate_rhs(s)
        cooling_time = abs(s["ge"] / rhs["ge"]) if rhs["ge"] != 0 else dtf_total
        dt = min(safety_factor * cooling_time, dtf_total - t)
        if dt <= 0:
            break
        converged, actual_t = solver.step_inplace(dt, niter=200, reltol=1e-5)
        if not converged:
            break
        t += actual_t
        s = state_scalar()
        t_hist.append(t)
        T_hist.append(float(solver.T[0]))
        h2_hist.append(h2_fraction(s))
        if t >= dtf_total:
            break
    return np.asarray(t_hist), np.asarray(T_hist), np.asarray(h2_hist), "Time (s)"


def run_freefall(nH, T, x_ion, x_h2, log_n_target, safety_factor=0.01, max_steps=5000):
    """Free-fall collapse from nH up to 10**log_n_target cm^-3 (same
    plain free-fall prescription as examples/free_fall_collapse.py)."""
    set_state(make_ics(nH, T, x_ion, x_h2))
    n_target = 10.0 ** log_n_target
    n_current = nH
    n_hist, T_hist, h2_hist = [], [], []
    for _ in range(max_steps):
        if n_current >= n_target:
            break
        rho = n_current * MH
        t_ff = np.sqrt(3.0 * np.pi / (32.0 * G_GRAV * rho))
        dt = safety_factor * t_ff
        rho_new = (rho ** -0.5 - np.sqrt(32.0 * G_GRAV / (3.0 * np.pi)) * dt) ** -2.0
        density_ratio = rho_new / rho

        for name in species_names:
            if name != "ge":
                solver.state[:, mod.SPECIES_INDEX[name]] *= density_ratio
        gamma_ad = thermodynamic_gamma(state_scalar())
        solver.state[:, mod.SPECIES_INDEX["ge"]] *= (1.0 + (gamma_ad - 1.0) * (density_ratio - 1.0))

        converged, _ = solver.step_inplace(dt, niter=200, reltol=1e-5)
        if not converged:
            break
        s = state_scalar()
        n_current = sum(s[n] for n in species_names if n not in ("ge", "de"))
        n_hist.append(n_current)
        T_hist.append(float(solver.T[0]))
        h2_hist.append(h2_fraction(s))
    return np.asarray(n_hist), np.asarray(T_hist), np.asarray(h2_hist), "n (cm$^{-3}$)"

In [ ]:
slider_layout = widgets.Layout(width="420px")
common = dict(continuous_update=False, layout=slider_layout, style={"description_width": "140px"})

w_nH = widgets.FloatLogSlider(value=1e4, base=10, min=-2, max=17, step=0.25, description="nₕ (cm⁻³)", **common)
w_T = widgets.FloatLogSlider(value=1000, base=10, min=1, max=4.7, step=0.05, description="T (K)", **common)
w_xion = widgets.FloatLogSlider(value=1e-4, base=10, min=-8, max=0, step=0.25, description="ionized fraction", **common)
w_xh2 = widgets.FloatLogSlider(value=1e-6, base=10, min=-10, max=0, step=0.25, description="H2 fraction", **common)
w_mode = widgets.ToggleButtons(options=["Cool at constant density", "Free-fall collapse"], description="Mode")
w_dtf = widgets.FloatLogSlider(value=13, base=10, min=6, max=17, step=0.25, description="total time: 10^x s", **common)
w_ntarget = widgets.FloatLogSlider(value=15, base=10, min=2, max=18, step=0.25, description="target n: 10^x cm⁻³", **common)

out = widgets.Output()


def on_mode_change(change):
    freefall = w_mode.value == "Free-fall collapse"
    w_ntarget.layout.display = "" if freefall else "none"
    w_dtf.layout.display = "none" if freefall else ""


def update(change=None):
    with out:
        out.clear_output(wait=True)
        try:
            if w_mode.value == "Free-fall collapse":
                x, T, h2, xlabel = run_freefall(w_nH.value, w_T.value, w_xion.value, w_xh2.value, w_ntarget.value)
            else:
                x, T, h2, xlabel = run_constant_density(w_nH.value, w_T.value, w_xion.value, w_xh2.value, w_dtf.value)
        except Exception as exc:  # noqa: BLE001 -- surface any solve failure in the output, not a dead kernel
            print(f"Solve failed: {exc}")
            return
        if len(x) == 0:
            print("No steps completed -- try a less extreme starting point.")
            return

        fig, (ax_T, ax_h2) = plt.subplots(2, 1, figsize=(6.5, 5.5), sharex=True)
        ax_T.loglog(x, T)
        ax_T.axhspan(1500, 2500, color="orange", alpha=0.15, label="H2-formation-heating target range")
        ax_T.set_ylabel("Temperature (K)")
        ax_T.legend(loc="upper left", fontsize=8)
        ax_T.set_title(f"final: T = {T[-1]:.1f} K, H2/H_tot = {h2[-1]:.3e}, {len(x)} steps")

        ax_h2.loglog(x, h2)
        ax_h2.set_ylabel("H2 / H_tot")
        ax_h2.set_xlabel(xlabel)

        fig.tight_layout()
        plt.show()


for w in (w_nH, w_T, w_xion, w_xh2, w_dtf, w_ntarget):
    w.observe(update, names="value")
w_mode.observe(on_mode_change, names="value")
w_mode.observe(update, names="value")
on_mode_change({"new": w_mode.value})

ui = widgets.VBox([
    widgets.HBox([w_nH, w_T]),
    widgets.HBox([w_xion, w_xh2]),
    w_mode,
    w_dtf,
    w_ntarget,
])
display(widgets.VBox([ui, out]))
update()

In [ ]:
# Close the persistent Solver handle when you're done with the notebook
# (also happens automatically on kernel shutdown/garbage collection, but
# not necessarily promptly).
# solver.close()